In [9]:
import os
from sqlalchemy import create_engine, text

# MySQL Connection Parameters
DB_USER = "root"
DB_PASSWORD = "stephen0111301468" 
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "kenya_maize_db"

#defining the path
sql_file_path = "database_schema.sql"

if not os.path.exists(sql_file_path):
    print(f"Error:Could not find {sql_file_path} in the directory.")
    
print("Connecting to MySQL server...")
engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}")
print(f"Reading and ececuting queries from {sql_file_path}...")
with open(sql_file_path,"r") as file:
    sql_script = file.read()
#execute statements block by block using semicolons
with engine.begin() as connection:
    for statement in sql_script.split(";"):
        cleaned_statement = statement.strip()
        if cleaned_statement:
            connection.execute(text(cleaned_statement))


print("SUCCESS: Relational database and tables created successfully from the SQL file!")



Connecting to MySQL server...
Reading and ececuting queries from database_schema.sql...
SUCCESS: Relational database and tables created successfully from the SQL file!


In [10]:
# Inspect columns of maize_prices table
import pandas as pd
inspect_query = text("DESCRIBE maize_prices;")
df_schema = pd.read_sql(inspect_query, engine)
print(df_schema)

                     Field           Type Null  Key            Default  \
0                 price_id            int   NO  PRI                NaN   
1           reporting_date           date   NO  MUL                NaN   
2                market_id            int   NO  MUL                NaN   
3                source_id            int   NO                     NaN   
4  wholesale_price_cleaned  decimal(10,2)   NO                     NaN   
5      retail_price_per_kg  decimal(10,2)  YES                     NaN   
6     supply_volume_tonnes  decimal(10,2)  YES                     NaN   
7        standardized_unit    varchar(50)  YES                90kg bag   
8               CREATED_AT      timestamp  YES       CURRENT_TIMESTAMP   

               Extra  
0     auto_increment  
1                     
2                     
3                     
4                     
5                     
6                     
7                     
8  DEFAULT_GENERATED  


In [11]:
query_1 = text("""
    SELECT 
        c.county_name,
        m.market_name,
        p.reporting_date,
        p.wholesale_price_cleaned,
        p.retail_price_per_kg,
        p.supply_volume_tonnes,
        p.standardized_unit
    FROM maize_prices p
    JOIN markets m ON p.market_id = m.market_id
    JOIN counties c ON m.county_id = c.county_id
    ORDER BY p.reporting_date DESC, c.county_name ASC
    LIMIT 10;
""")

import pandas as pd
df_query1 = pd.read_sql(query_1, engine)
print(df_query1)

Empty DataFrame
Columns: [county_name, market_name, reporting_date, wholesale_price_cleaned, retail_price_per_kg, supply_volume_tonnes, standardized_unit]
Index: []
